# CNN propia (baseline)

Brain Tumor Probability (Binary Classification)
- Yes: tumor presente
- No: tumor ausente

In [2]:
# pip install scikit-learn
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Model
from pathlib import Path

## Hyperparameters (Model Configuration)

| Parameter | Value / Configuration | Rationale / Reference |
| --- | --- | --- |
| **Input Dimensions** | 256 | 256 x 256 pixels square resolution for spatial consistency. |
| **Batch Size** | 8 | Set in accordance with Angelina et al. (2026, p. 9) for memory constraints. |
| **Epochs** | 100 | Sufficient horizon to ensure convergence. |
| **Learning Rate** | 0.0001 (Static) | Set in accordance with He et al. (2015, p. 4). |
| **Dropout Rate** | 0.5 | Applied for regularization to mitigate overfitting (Hinton et al., 2012, p. 2). |
| **Activation Function** | Sigmoid | Employed in the final layer for single-class probability mapping. |
| **Loss Function** | Binary Cross-Entropy | Selected to complement the single-neuron sigmoid output. |
| **Optimizer** | Adam | Back propagation optimization. |
| **Data Augmentation** | Rotation, translation, zoom, and contrast | Applied only to training batches to reduce overfitting in the small dataset. |

- He, K., Zhang, X., Ren, S., y Sun, J. (2015). Deep residual learning for image recognition. arXiv. https://doi.org/10.48550/arXiv.1512.03385
- Hinton, G. E., Srivastava, N., Krizhevsky, A., Sutskever, I., y Salakhutdinov, R. R. (2012). Improving neural networks by preventing co-adaptation of feature detectors. arXiv. https://doi.org/10.48550/arXiv.1207.0580
- Angelina, C. L., Xiao, F.-R., Vyas, S., Yang, P.-C., Chang, H.-T., & Luo, Y. (2026). Mod-SE(2): A geometric deep learning framework for brain tumor classification and segmentation in MRI images. Journal of Biomedical Science, 33, Article 11. https://doi.org/10.1186/s12929-025-01213-y

In [12]:
IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 100
LEARNING_RATE = 0.01
DROPOUT = 0.5
LOSS_METHOD = "binary_crossentropy"
ACTIVATION = "sigmoid"
LABEL_MODE = "binary"
PATIENCE = 15
SEED = 42

THRESHOLD = 0.5

TRAIN_DIR = "data/dataset_ready/train"
VAL_DIR = "data/dataset_ready/val"
TEST_DIR = "data/dataset_ready/test"

MODEL_TAG = "cnn_propia_03"

## Dataset Structure
```text
data/dataset_ready/
+-- train/
|   +-- no/
|   +-- yes/
+-- val/
|   +-- no/
|   +-- yes/
+-- test/
    +-- no/
    +-- yes/
```

En TensorFlow/Keras, la asignacion de etiquetas para conjuntos de datos estructurados en carpetas locales no requiere un etiquetado manual en codigo. `image_dataset_from_directory()` infiere las clases por el nombre de las carpetas y las ordena alfabeticamente: `no -> 0`, `yes -> 1`.

El conjunto de **validation** se usa correctamente cuando se pasa como `validation_data=val_ds` en `model.fit()`: sirve para monitorear el entrenamiento, seleccionar el mejor checkpoint y activar early stopping. No debe recibir data augmentation ni debe usarse para reportar el resultado final. El conjunto de **test** se mantiene separado hasta el final para medir el desempeno del modelo ya entrenado.

In [4]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode=LABEL_MODE,
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode=LABEL_MODE,
    shuffle=False
)

classes = train_ds.class_names # Binary: ['no', 'yes']
num_classes = len(classes)
print(classes)

Found 151 files belonging to 2 classes.
Found 51 files belonging to 2 classes.
['no', 'yes']


## Data Augmentation

Se agrega data augmentation para generar variaciones realistas de las imagenes de entrenamiento sin crear archivos nuevos en disco. En cada epoca, Keras aplica pequenas rotaciones, traslaciones, zoom y cambios de contraste a los batches de `train_ds`.

Esto busca que el modelo no memorice tan facil el dataset pequeno y aprenda patrones mas robustos. La validacion y el test se dejan sin augmentation para medir el desempeno sobre imagenes reales, no transformadas artificialmente.

For the classification head, the two 4096-unit dense layers were removed, as the task involved only two output classes (N=256), making such a high-capacity head unnecessarily complex.
| Block       | Architecture                                |
|-------------|---------------------------------------------|
| **VGG19 Layer 1** | Conv(64) → Conv(64) → MaxPool               |
| **VGG19 Layer 2** | Conv(128) → Conv(128) → MaxPool             |
| **VGG19 Layer 3**| Conv(256) → Conv(256) → Conv(256) → MaxPool |
| **VGG19 Layer 4** | Conv(512) → Conv(512) → Conv(512) → MaxPool |
| **VGG19 Layer 5** | Conv(512) → Conv(512) → Conv(512) → MaxPool |
| **Head (manual)** | Flatten → Dense(1000) → Softmax |
- He, K., Zhang, X., Ren, S., y Sun, J. (2015). Deep residual learning for image recognition. arXiv. https://doi.org/10.48550/arXiv.1512.03385, page 4

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense, Input, Dropout, MaxPooling2D, Conv2D
from tensorflow.keras.layers import RandomRotation, RandomTranslation, RandomZoom, RandomContrast

# Data augmentation dentro del modelo: se activa en training y se desactiva en validation/test.
data_augmentation = Sequential(
    [
        RandomRotation(0.05, fill_mode="nearest"),
        RandomTranslation(0.05, 0.05, fill_mode="nearest"),
        RandomZoom(0.10, fill_mode="nearest"),
        RandomContrast(0.10),
    ],
    name="data_augmentation"
)

# Modelo inspirado de VGG19 https://doi.org/10.48550/arXiv.1512.03385
model = Sequential()
model.add(Input(shape=(IMG_SIZE, IMG_SIZE, 3)))
model.add(data_augmentation)

# Block 1
model.add(Conv2D(32, kernel_size=3, activation="relu", padding="same"))
model.add(Conv2D(32, kernel_size=3, activation="relu", padding="same"))
model.add(MaxPooling2D(pool_size=(2, 2)))

# Block 2
model.add(Conv2D(32, kernel_size=3, activation="relu", padding="same"))
model.add(Conv2D(32, kernel_size=3, activation="relu", padding="same"))
model.add(MaxPooling2D(pool_size=(2, 2)))

# Head
model.add(Flatten())
model.add(Dense(32, activation="relu"))
model.add(Dropout(DROPOUT))

# Output
model.add(Dense(1, activation=ACTIVATION)) # Binary = 1
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ data_augmentation (Sequential)  │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 256, 256, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 256, 256, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 128, 128, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 128, 128, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 131072)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     8,388,672 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,417,377 (32.11 MB)

 Trainable params: 8,417,377 (32.11 MB)

 Non-trainable params: 0 (0.00 B)

## Compile

In [13]:

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=LEARNING_RATE
    ),
    loss=LOSS_METHOD,
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.F1Score(name="f1score", threshold=THRESHOLD)
    ]
)

## Train

Ejemplo de overfitting: 

Epoch 1
train acc = 0.80,
val acc   = 0.78

Epoch 5
train acc = 0.95,
val acc   = 0.90

Epoch 10
train acc = 0.99,
val acc   = 0.75

In [ ]:
# 5 Minutes
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        f"checkpoints/{MODEL_TAG}/{MODEL_TAG}.keras",
        save_best_only=True,
        monitor="val_f1score" # val_ prefijo automático de Keras, f1score de validation no de train
    ),
    tf.keras.callbacks.EarlyStopping(
        patience=PATIENCE,
        restore_best_weights=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds, 
    epochs=EPOCHS,
    callbacks=callbacks
)

Epoch 1/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 24s 690ms/step - accuracy: 0.4636 - auc: 0.4283 - f1score: 0.5424 - loss: 245.7267 - precision: 0.5714 - recall: 0.5161 - val_accuracy: 0.6078 - val_auc: 0.6274 - val_f1score: 0.7561 - val_loss: 0.6812 - val_precision: 0.6078 - val_recall: 1.0000
Epoch 2/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 12s 649ms/step - accuracy: 0.6093 - auc: 0.4872 - f1score: 0.7572 - loss: 0.6818 - precision: 0.6133 - recall: 0.9892 - val_accuracy: 0.6078 - val_auc: 0.6145 - val_f1score: 0.7561 - val_loss: 0.6490 - val_precision: 0.6078 - val_recall: 1.0000
Epoch 3/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 23s 789ms/step - accuracy: 0.6159 - auc: 0.5180 - f1score: 0.7623 - loss: 0.6708 - precision: 0.6159 - recall: 1.0000 - val_accuracy: 0.6078 - val_auc: 0.5581 - val_f1score: 0.7561 - val_loss: 0.6358 - val_precision: 0.6078 - val_recall: 1.0000
Epoch 4/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 15s 791ms/step - accuracy: 0.6159 - auc: 0.5825 - f1score: 0.7623 - loss: 0.6649 - precision: 0.6159 - reca

## Training Curves

In [1]:
reportPath = os.path.join("reports", MODEL_TAG)
Path(reportPath).mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(10,4))
plt.plot(history.history["accuracy"])
plt.plot(history.history["val_accuracy"])
plt.legend(["Train","Validation"])
plt.title("Accuracy")
plt.savefig(os.path.join(reportPath, "accuracy.png"), dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(10,4))
plt.plot(history.history["loss"][1:]) # Remueve el primer elemento (aplastamiento visual de la grafica)
plt.plot(history.history["val_loss"][1:]) # Remueve el primer elemento (aplastamiento visual de la grafica)

plt.legend(["Train","Validation"])
plt.title("Loss")
plt.savefig(os.path.join(reportPath, "loss.png"), dpi=300, bbox_inches="tight")
plt.show()

NameError: name 'os' is not defined

## Save Model is already saved based on f1score performance